# Market Basket Analysis

---

**Author:** Raditya Zaki Athaya   
**Notebook:** `03_market_basket_analysis.ipynb`

---

## Project Overview

Market Basket Analysis (MBA) bertujuan menemukan pola asosiasi antar produk yang sering dibeli bersamaan dalam satu transaksi. Dengan menggunakan algoritma **Apriori**, kita dapat mengidentifikasi *association rules* seperti:

> *"Pelanggan yang membeli produk A cenderung juga membeli produk B."*

Output analisis ini digunakan untuk dua tujuan bisnis utama:
1. **Cross-selling:** Merekomendasikan produk pelengkap secara otomatis di halaman checkout.
2. **Bundle Promotion:** Merancang paket produk dengan harga khusus berdasarkan kombinasi yang terbukti sering dibeli bersamaan.

## 1. Import Library & Load Data

Selain library standar, notebook ini menggunakan `mlxtend` — library machine learning yang menyediakan implementasi algoritma Apriori dan fungsi `association_rules` siap pakai.

Pastikan library sudah terinstall:
```bash
pip install mlxtend
```

In [2]:
# ── Standard Libraries ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

# ── mlxtend: Apriori & Association Rules ─────────────────────────────────────
from mlxtend.frequent_patterns import apriori, association_rules

# ── Global Config ────────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

# ── Load Cleaned Dataset ─────────────────────────────────────────────────────
CLEANED_PATH = '../data/processed/ecommerce_cleaned.csv'

df = pd.read_csv(CLEANED_PATH, dtype={'CustomerID': str, 'InvoiceNo': str})
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='mixed')

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique countries: {df['Country'].nunique()}")
print(f"Unique invoices : {df['InvoiceNo'].nunique():,}")
df.head()

Dataset loaded: 392,692 rows x 13 columns
Unique countries: 37
Unique invoices : 18,532


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSpend,Year,Month,DayOfWeek,Hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.5500,17850,United Kingdom,15.3000,2010,12,Wednesday,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.3900,17850,United Kingdom,20.3400,2010,12,Wednesday,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.7500,17850,United Kingdom,22.0000,2010,12,Wednesday,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.3900,17850,United Kingdom,20.3400,2010,12,Wednesday,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.3900,17850,United Kingdom,20.3400,2010,12,Wednesday,8


## 2. Data Preparation (Basket Matrix)

Untuk analisis ini, kita fokus pada transaksi dari **United Kingdom** saja, dengan dua alasan:

1. **Efisiensi memori** — UK mendominasi ~90% transaksi. Memasukkan semua negara akan menghasilkan matriks yang sangat sparse dan lambat diproses.
2. **Spesifisitas pasar** — Perilaku pembelian antar negara bisa berbeda. Fokus pada satu pasar menghasilkan rules yang lebih relevan dan dapat ditindaklanjuti.

Langkah transformasi:
- `groupby` invoice dan produk → agregasi total quantity
- `unstack` → pivot menjadi *basket matrix* (baris = invoice, kolom = produk)
- Encode: semua nilai ≥ 1 → `1` (dibeli), nilai ≤ 0 → `0` (tidak dibeli)

In [3]:
# ── Filter: Hanya Transaksi United Kingdom ────────────────────────────────────
df_uk = df[df['Country'] == 'United Kingdom'].copy()

print(f"Transaksi UK : {len(df_uk):,} rows")
print(f"Unique invoices UK: {df_uk['InvoiceNo'].nunique():,}")

# ── Buat Basket Matrix ────────────────────────────────────────────────────────
# Baris = InvoiceNo, Kolom = nama produk (Description), Nilai = total Quantity
basket = (
    df_uk
    .groupby(['InvoiceNo', 'Description'])['Quantity']
    .sum()
    .unstack(fill_value=0)
)

print(f"\nBasket matrix shape: {basket.shape}")
print(f"  → {basket.shape[0]:,} invoices x {basket.shape[1]:,} unique products")

# ── Encode: Ubah Quantity menjadi Binary (0/1) ────────────────────────────────
def encode_units(x):
    # Quantity <= 0 → 0 (tidak dibeli / retur)
    # Quantity >= 1 → 1 (dibeli)
    return 0 if x <= 0 else 1

basket_encoded = basket.map(encode_units)

# Verifikasi: pastikan hanya ada nilai 0 dan 1
assert basket_encoded.isin([0, 1]).all().all(), "Masih ada nilai selain 0 dan 1!"

print(f"\nEncoding selesai. Sparsity matrix:")
total_cells  = basket_encoded.shape[0] * basket_encoded.shape[1]
filled_cells = basket_encoded.sum().sum()
print(f"  Cells berisi 1 : {filled_cells:,} ({filled_cells/total_cells*100:.2f}%)")
print(f"  Cells berisi 0 : {total_cells - filled_cells:,} ({(total_cells-filled_cells)/total_cells*100:.2f}%)")

Transaksi UK : 349,203 rows
Unique invoices UK: 16,646

Basket matrix shape: (16646, 3844)
  → 16,646 invoices x 3,844 unique products

Encoding selesai. Sparsity matrix:
  Cells berisi 1 : 344,342 (0.54%)
  Cells berisi 0 : 63,642,882 (99.46%)
